# RGP-13: 초임계 연소 해석을 위한 실유체 모델 구현 가이드
## OpenFOAM-13 기반 SRK / Chung / Takahashi 구현 계획

### 목표
Oefelein의 초임계 연소 모델링 프레임워크를 OpenFOAM-13에 구현한다.

### 참조
- **Oefelein, J.C.** — "Mixing and combustion of cryogenic oxygen-hydrogen shear-coaxial jet flames at supercritical pressure" (Combustion Science and Technology, 2006)
- **Chung, T.H. et al.** — "Generalized Multiparameter Correlation for Nonpolar and Polar Fluid Transport Properties" (Ind. Eng. Chem. Res., 1988)
- **Takahashi, S.** — "Preparation of a Generalized Chart for the Diffusion Coefficients of Gases at High Pressures" (J. Chem. Eng. Japan, 1975)
- **realFluidFoam-8** (UNIST, Prof. Chun Sang Yoo 연구실) — OpenFOAM v8 기반 구현체
- **OpenFOAM-13** `PengRobinsonGas` — 기존 3차 상태방정식 참조 패턴

### 구현 범위

| 구성요소 | 모델 | 역할 |
|---------|------|------|
| **상태방정식 (EOS)** | SRK (Soave-Redlich-Kwong) | ρ, h, Cp, Cv, Z 등 열역학 물성 |
| **점성/열전도도** | Chung et al. (1988) | μ(p,T), κ(p,T) 고압 보정 |
| **확산계수** | Takahashi (1975) + Fuller | D_mix(p,T) 고압 보정 |
| **혼합 규칙** | 2차 결합 규칙 (quadratic mixing) | 다성분 혼합물 물성 계산 |

In [ ]:
from IPython.display import display, HTML

def render_mermaid(mermaid_code, width="100%", height=None):
    import hashlib
    uid = "m" + hashlib.md5(mermaid_code.encode()).hexdigest()[:8]
    height_style = f"min-height:{height}px;" if height else ""
    html = f"""
    <div id="{uid}" style="width:{width};{height_style}">
        <pre class="mermaid">
{mermaid_code}
        </pre>
    </div>
    <script type="module">
        import mermaid from 'https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs';
        mermaid.initialize({{ startOnLoad: true, theme: 'base',
            themeVariables: {{
                primaryColor: '#BBDEFB', primaryTextColor: '#1a1a1a',
                primaryBorderColor: '#1565C0', lineColor: '#555',
                secondaryColor: '#C8E6C9', tertiaryColor: '#FFF3E0',
                fontSize: '14px'
            }}
        }});
        mermaid.run({{ nodes: document.querySelectorAll('#{uid} .mermaid') }});
    </script>
    """
    display(HTML(html))

---
## 1. 아키텍처 개요: OpenFOAM-13 vs realFluidFoam-8 차이점

### OpenFOAM-13 템플릿 조합 체인
```
Transport< species::thermo< Thermo< EOS< Specie > >, Energy > >
```

예시: `sutherlandTransport< species::thermo< janafThermo< perfectGas<specie> >, sensibleEnthalpy > >`

### 핵심 차이: OF-13 vs OFv8

| 항목 | OFv8 (realFluidFoam-8) | OF-13 (대상) |
|------|----------------------|-------------|
| **Specie 기본 클래스** | `rfSpecie` (Tc, Pc, Vc, omega, miui, kappai, sigmvi 추가) | `specie` (molWeight, Y만 보유) |
| **혼합 모델** | `SRKchungTakaMixture` 전용 클래스 | `multicomponentMixture` + operator 기반 |
| **EOS 혼합** | Mixture 클래스 내 `updateEoS()` 직접 호출 | operator+=로 물성 합산 후 사용 |
| **Transport 혼합** | Mixture 클래스 내 `updateTRANS()` 직접 호출 | operator+=로 물성 합산 |
| **매크로 체계** | `forSRKchungTakaGases.H` 전용 매크로 | `forGases.H` 범용 매크로 확장 |
| **인스턴스화** | 별도 라이브러리로 컴파일 | specie 라이브러리에 통합 |

### 구현 전략: 두 가지 접근법

**접근법 A — 최소 침습 (권장):**  
기존 OF-13 프레임워크에 플러그인 형태로 추가. `specie` 기본 클래스를 확장한 `rfSpecie` 도입 후, SRK/Chung/Takahashi를 각각 독립 모듈로 구현. 전용 `SRKchungTakaMixture` 혼합 클래스 생성.

**접근법 B — 순수 OF-13 스타일:**  
operator 기반 혼합만 사용. 그러나 Chung/Takahashi의 복잡한 2차 결합 규칙을 operator에 넣기 어려움. **비권장.**

In [ ]:
render_mermaid("""
graph TD
    subgraph OF13["OpenFOAM-13 기존 구조"]
        direction TB
        SP13["specie\n(molWeight, Y)"]
        EOS13["PengRobinsonGas\n(Tc, Vc, Zc, Pc, omega)"]
        TH13["janafThermo\n(JANAF 다항식)"]
        TR13["sutherlandTransport\n(mu, kappa)"]
        SP13 --> EOS13 --> TH13 --> TR13
    end

    subgraph RGP13["RGP-13 신규 구현"]
        direction TB
        RFSP["<b>rfSpecie</b>\n(specie 확장)\nTc, Pc, Vc, omega\nmiui, kappai, sigmvi"]
        SRK["<b>SRKGas</b>\n(SRK 상태방정식)\nb, coef1~3, Z solver"]
        JANAF2["janafThermo\n(기존 재사용)"]
        CHUNG["<b>chungTransport</b>\nmu: Chung (1988)\nkappa: Chung (1988)\nDimix: Takahashi (1975)"]
        RFSP --> SRK --> JANAF2 --> CHUNG
    end

    subgraph MIXTURE["전용 혼합 클래스"]
        MIX["<b>SRKchungTakaMixture</b>\nSRK: 2차 결합규칙 (a, b)\nChung: L-J 파라미터 혼합\nTakahashi: 이성분 확산 혼합"]
    end

    TR13 -.->|"기존 체인"| MM13["multicomponentMixture\n(operator 기반)"]
    CHUNG -->|"신규 체인"| MIX

    style OF13 fill:#F5F5F5,stroke:#9E9E9E
    style RGP13 fill:#E3F2FD,stroke:#1565C0
    style MIXTURE fill:#FFF3E0,stroke:#E65100
    style RFSP fill:#BBDEFB,stroke:#1565C0
    style SRK fill:#C8E6C9,stroke:#2E7D32
    style CHUNG fill:#FFCCBC,stroke:#E64A19
    style MIX fill:#FFE0B2,stroke:#E65100
""", height=500)

---
## 2. 구현 단계 총괄 로드맵

In [ ]:
render_mermaid("""
gantt
    title RGP-13 구현 로드맵
    dateFormat YYYY-MM-DD
    axisFormat %m/%d

    section Phase 1: 기반
    rfSpecie 기본 클래스 확장           :p1a, 2026-04-07, 3d
    SRKGas 상태방정식 구현              :p1b, after p1a, 5d
    단일종 검증 (NIST 비교)             :p1c, after p1b, 3d

    section Phase 2: 수송물성
    chungTransport (mu, kappa)         :p2a, after p1c, 5d
    Takahashi 확산계수 (Dimix)          :p2b, after p2a, 4d
    단일종 수송물성 검증                 :p2c, after p2b, 3d

    section Phase 3: 혼합규칙
    SRKchungTakaMixture 클래스          :p3a, after p2c, 7d
    SRK 2차 결합규칙 구현               :p3b, after p2c, 5d
    Chung 파라미터 혼합규칙             :p3c, after p3b, 4d
    Takahashi 이성분 혼합규칙           :p3d, after p3c, 3d

    section Phase 4: 통합
    템플릿 인스턴스화 매크로             :p4a, after p3a, 3d
    multicomponentFluid 솔버 연동       :p4b, after p4a, 5d
    SandiaD_LTS 초임계 변환 테스트      :p4c, after p4b, 5d

    section Phase 5: 검증
    NIST 다성분 물성 비교               :p5a, after p4c, 4d
    Oefelein LOx/GH2 벤치마크           :p5b, after p5a, 7d
    문서화                              :p5c, after p5b, 3d
""")

---
## 3. Phase 1 — rfSpecie 기본 클래스 + SRK 상태방정식

### 3.1 rfSpecie: 실유체 종 기본 클래스

`specie`를 확장하여 초임계 물성 계산에 필요한 임계점 물성을 저장한다.

**파일 위치:** `src/thermophysicalModels/specie/rfSpecie/`

```
rfSpecie/
├── rfSpecie.H      ← 클래스 선언
├── rfSpecieI.H     ← 인라인 구현
└── rfSpecie.C      ← 딕셔너리 생성자, I/O
```

**추가 멤버 변수 (specie 대비):**

| 변수 | 타입 | 단위 | 설명 |
|------|------|------|------|
| `Tc_` | scalar | K | 임계 온도 |
| `Pc_` | scalar | Pa | 임계 압력 |
| `Vc_` | scalar | cm³/mol | 임계 부피 |
| `omega_` | scalar | - | 이심인자 (acentric factor) |
| `miui_` | scalar | Debye | 쌍극자 모멘트 |
| `kappai_` | scalar | - | 결합 인자 (association factor) |
| `sigmvi_` | scalar | - | 확산 부피 (Fuller) |

**딕셔너리 입력 형식:**
```cpp
specie
{
    molWeight   16.043;  // CH4
}
rfProperties
{
    Tc          190.56;
    Pc          4.599e6;
    Vc          98.6;
    omega       0.0115;
    miui        0.0;
    kappai      0.0;
    sigmvi      24.42;
}
```

**구현 시 주의사항:**
- `operator+=`에서 mole-fraction 기반 선형 혼합 (Tc, Pc, Vc, omega)
- `operator*=`에서는 Specie만 스케일, 임계점 물성은 불변
- OFv8의 `rfSpecie`를 참조하되, OF-13의 `specie` 인터페이스에 맞게 조정

### 3.2 SRKGas: Soave-Redlich-Kwong 상태방정식

**파일 위치:** `src/thermophysicalModels/specie/equationOfState/SRKGas/`

```
SRKGas/
├── SRKGas.H       ← 클래스 선언 (PengRobinsonGas.H 패턴)
├── SRKGasI.H      ← 인라인 구현 (핵심 수식)
└── SRKGas.C       ← 딕셔너리 생성자, write()
```

**참조:** `PengRobinsonGas` (OF-13) + `soaveRedlichKwong` (realFluidFoam-8)

#### SRK 상태방정식 수식

$$P = \frac{RT}{V-b} - \frac{a \cdot \alpha(T)}{V(V+b)}$$

**순수 성분 파라미터:**

$$a = 0.42747 \frac{R^2 T_c^2}{P_c}, \quad b = 0.08664 \frac{R T_c}{P_c}$$

$$\alpha(T) = \left[1 + S\left(1 - \sqrt{T/T_c}\right)\right]^2$$

$$S = 0.48508 + 1.5517\omega - 0.15613\omega^2$$

**분해된 계수 형태 (realFluidFoam-8 방식):**

$$a\alpha(T) = \text{coef1} - \text{coef2}\sqrt{T} + \text{coef3} \cdot T$$

| 계수 | 수식 |
|------|------|
| coef1 | $a(1+S)^2$ |
| coef2 | $2aS(1+S)/\sqrt{T_c}$ |
| coef3 | $aS^2/T_c$ |

#### 3차 방정식 (압축인자 Z)

$$Z^3 - Z^2 + (A - B - B^2)Z - AB = 0$$

$$A = \frac{a\alpha \cdot p}{R^2 T^2}, \quad B = \frac{bp}{RT}$$

**Cardano 공식으로 풀이** — 3근 중 최대값 (가스상) 또는 최소값 (액상) 선택

#### 구현해야 할 메서드 (OF-13 인터페이스)

| 메서드 | 수식 | OFv8 구현 여부 |
|--------|------|---------------|
| `rho(p,T)` | $\rho = p/(ZRT)$ | O |
| `h(p,T)` | 출발 엔탈피 $H^{dep}$ | O |
| `Cp(p,T)` | 출발 Cp — M, N 항 포함 | O |
| `e(p,T)` | $e = h - p/\rho$ | **X (미구현 → 구현 필요)** |
| `Cv(p,T)` | $Cv = Cp - CpMCv$ | **X (미구현 → 구현 필요)** |
| `sp(p,T)` | 출발 엔트로피 | O |
| `sv(p,T)` | Cv 적분 엔트로피 | **X (미구현 → 구현 필요)** |
| `psi(p,T)` | $\psi = 1/(ZRT)$ | O |
| `Z(p,T)` | Cardano 3차 방정식 | O |
| `CpMCv(p,T)` | M, N 항 기반 | O |
| `alphav(p,T)` | 체적 팽창 계수 | **X (미구현 → 구현 필요)** |

> **핵심 개선점:** OFv8에서 미구현(return 0)인 `e`, `Cv`, `sv`, `alphav`를 OF-13에서는 완전 구현해야 한다.

### 3.3 SRK 출발 물성 (Departure Functions) — 미구현 메서드 보완 수식

OFv8에서 `return 0`으로 처리된 메서드들의 정확한 수식:

#### 내부 에너지 출발 함수 `e(p,T)`
$$e^{dep} = h^{dep} - (Z-1)RT = \frac{a\alpha - T \cdot da\alpha/dT}{b} \ln\left(\frac{Z+B}{Z}\right)$$

#### 정적 비열 출발 함수 `Cv(p,T)`
$$C_v^{dep} = C_p^{dep} - (C_p - C_v)^{dep} + R$$

또는 직접 계산:
$$C_v^{dep} = \frac{T \cdot d^2(a\alpha)/dT^2}{b} \ln\left(\frac{Z+B}{Z}\right)$$

#### 엔트로피 (Cv 적분) `sv(p,T)`
PengRobinsonGas 패턴 참조 — OF-13에서도 NotImplemented이므로 동일하게 처리 가능

#### 체적 팽창 계수 `alphav(p,T)`
$$\alpha_v = \frac{1}{V}\left(\frac{\partial V}{\partial T}\right)_p = -\frac{1}{\rho}\left(\frac{\partial \rho}{\partial T}\right)_p$$

수치 미분으로 구현 가능:
$$\alpha_v \approx -\frac{1}{\rho}\frac{\rho(p, T+\delta T) - \rho(p, T-\delta T)}{2\delta T}$$

---
## 4. Phase 2 — chungTransport: Chung 점성/열전도도 + Takahashi 확산계수

**파일 위치:** `src/thermophysicalModels/specie/transport/chungTransport/`

```
chungTransport/
├── chungTransport.H       ← 클래스 선언
├── chungTransportI.H      ← 인라인 구현 (mu, kappa, Dimix, phi)
└── chungTransport.C       ← 딕셔너리 생성자, I/O
```

### 4.1 Chung 점성 모델 (Chung et al., 1988)

#### 멤버 변수 (순수 성분 / 혼합물 공용)

| 변수 | 물리량 | 혼합규칙으로 업데이트 |
|------|--------|---------------------|
| `sigmaM_` | L-J 충돌 직경 [A] = $0.809 V_c^{1/3}$ | O |
| `epsilonkM_` | L-J 에너지/k [K] = $T_c / 1.2593$ | O |
| `MM_` | 분자량 [g/mol] | O |
| `VcM_` | 임계 부피 [cm³/mol] | O |
| `TcM_` | 임계 온도 [K] | O |
| `omegaM_` | 이심인자 | O |
| `miuiM_` | 쌍극자 모멘트 [D] | O |
| `kappaiM_` | 결합 인자 | O |

#### 중간 변수 계산 (`calculate()`)

$$T^* = T / (\epsilon/k)_M$$

$$\mu_r = \frac{131.3 \cdot \mu_i}{(V_c \cdot T_c)^{1/2}}$$

$$F_c = 1 - 0.2756\omega + 0.059035\mu_r^4 + \kappa_i$$

$$\Omega^* = AT^{*-B} + C \exp(-DT^*) + E \exp(-FT^*) + GT^{*B}\sin(ST^{*W} - H)$$

계수: A=1.16145, B=0.14874, C=0.52487, D=0.77320, E=2.16178, F=2.43787, G=-6.435e-4, H=7.27371, S=18.0323, W=-0.76830

$$\eta_0 = \frac{4.0785 \times 10^{-5} \sqrt{M \cdot T} \cdot F_c}{\Omega^* \cdot V_c^{2/3}} \quad [\text{poise}]$$

#### 점성 계산 (`mu(p,T)`)

$$Y = \frac{\rho V_c}{6}, \quad G_1 = \frac{1-0.5Y}{(1-Y)^3}$$

$$G_2 = \frac{A_1[1-\exp(-A_3 Y)]/Y + A_2 G_1 \exp(A_4 Y) + A_2 G_1}{A_1 A_3 + A_1 + A_2}$$

$$\eta_k = \eta_0\left(\frac{1}{G_2} + A_5 Y\right), \quad \eta_p = \frac{40.785\times10^{-6}}{\sqrt{T^*}} \cdot \frac{\sqrt{M T}}{V_c^{2/3}} \cdot A_6 Y^2 G_2 \exp\left(A_7 + \frac{A_8}{T^*} + \frac{A_9}{T^{*2}}\right)$$

$$\mu = 0.1(\eta_k + \eta_p) \quad [\text{kg/m}\cdot\text{s}]$$

계수 $A_i$는 4개의 테이블(a0, a1, a2, a3)에서 $\omega$, $\mu_r^4$, $\kappa_i$로 보간:
$$A_i = a_{0,i} + a_{1,i}\omega + a_{2,i}\mu_r^4 + a_{3,i}\kappa_i$$

### 4.2 Chung 열전도도 (`kappa(p,T)`)

$$\alpha = C_v/R - 3/2, \quad \beta = 0.7862 - 0.7109\omega + 1.1368\omega^2$$

$$T_r = T/T_c, \quad \mathcal{Z} = 2.0 + 10.5T_r^2$$

$$\Psi = 1 + \alpha \cdot \frac{0.215 + 0.28288\alpha - 1.061\beta + 0.26665\mathcal{Z}}{0.6366 + \beta\mathcal{Z} + 1.061\alpha\beta}$$

$$\lambda_0 = 7.452 \frac{\eta_0}{M} \Psi \quad [\text{cal/cm}\cdot\text{s}\cdot\text{K}]$$

$$H_2 = \frac{B_1[1-\exp(-B_3 Y)]/Y + B_2 G_1 \exp(B_4 Y) + B_2 G_1}{B_1 B_3 + B_1 + B_2}$$

$$\lambda_k = \lambda_0\left(\frac{1}{H_2} + B_5 Y\right), \quad \lambda_p = \frac{3.039\times10^{-4}(T_c/M)^{1/2}}{V_c^{2/3}} B_6 Y^2 H_2 \sqrt{T_r}$$

$$\kappa = 418.68(\lambda_k + \lambda_p) \quad [\text{W/m}\cdot\text{K}]$$

계수 $B_i = b_{0,i} + b_{1,i}\omega + b_{2,i}\mu_r^4 + b_{3,i}\kappa_i$

### 4.3 Takahashi 확산계수 (`Dimix(speciei, p, T)`)

Fuller 이론 + Takahashi 고압 보정:

$$D_{ij} = \phi(P_r, T_r) \cdot \frac{0.001 \cdot T^{1.75} \cdot M_{ij}^{0.5}}{(p/101325) \cdot \sigma_{ij}^2} \quad [\text{cm}^2/\text{s}]$$

$$M_{ij} = \frac{2 M_i M_j}{M_i + M_j}, \quad \sigma_{ij} = (\sigma_{v,i})^{1/3} + (\sigma_{v,j})^{1/3}$$

$$P_r = p/P_{c,ij}, \quad T_r = T/T_{c,ij}$$

혼합-평균 확산계수:
$$D_{i,\text{mix}} = \frac{1 - Y_i}{\sum_{j \neq i} X_j / D_{ij}}$$

$\phi(P_r, T_r)$ 함수: 16개 환산압력 구간에 대한 보간 테이블 (realFluidFoam-8의 `phi()` 함수 참조)

---
## 5. Phase 3 — SRKchungTakaMixture 혼합 규칙

이 단계가 가장 핵심이다. OF-13의 `multicomponentMixture`는 단순 operator 기반 혼합만 지원하므로, 
초임계 물성에 필요한 **2차 결합규칙(quadratic mixing rules)**을 구현하려면 전용 Mixture 클래스가 필수적이다.

**파일 위치:** `src/thermophysicalModels/multicomponentThermo/mixtures/SRKchungTakaMixture/`

```
SRKchungTakaMixture/
├── SRKchungTakaMixture.H      ← 클래스 선언
├── SRKchungTakaMixture.C      ← 혼합규칙 구현 (핵심)
└── SRKchungTakaMixtureI.H     ← 인라인 접근자
```

### 5.1 SRK 파라미터 혼합 규칙

#### 사전 계산 행렬 (초기화 시 1회)

각 종 쌍 (i, j)에 대해:

$$a_i = 0.42747 \frac{R^2 T_{c,i}^2}{P_{c,i}}, \quad S_i = 0.48508 + 1.5517\omega_i - 0.15613\omega_i^2$$

| 행렬 | 결합 규칙 |
|------|-----------|
| $\text{COEF1}_{ij}$ | $\sqrt{a_i a_j}(1+S_i)(1+S_j)$ |
| $\text{COEF2}_{ij}$ | $\sqrt{a_i a_j}\left[\frac{S_i(1+S_j)}{\sqrt{T_{c,i}}} + \frac{S_j(1+S_i)}{\sqrt{T_{c,j}}}\right]$ |
| $\text{COEF3}_{ij}$ | $\sqrt{a_i a_j} \cdot \frac{S_i S_j}{\sqrt{T_{c,i} T_{c,j}}}$ |
| $b_i$ | $0.08664 R T_{c,i}/P_{c,i}$ (선형) |

#### 런타임 혼합 (매 셀마다)

몰분율 $X_i$로부터:

$$b_M = \sum_i X_i b_i$$

$$\text{coef1}_M = \sum_i \sum_j X_i X_j \text{COEF1}_{ij}$$

$$\text{coef2}_M = \sum_i \sum_j X_i X_j \text{COEF2}_{ij}$$

$$\text{coef3}_M = \sum_i \sum_j X_i X_j \text{COEF3}_{ij}$$

> **참고:** 현재 구현에서는 이성분 상호작용 매개변수 $k_{ij} = 0$. 향후 $\sqrt{a_i a_j}(1-k_{ij})$ 형태로 확장 가능.

### 5.2 Chung 파라미터 혼합 규칙

#### 사전 계산 행렬

| 행렬 | 결합 규칙 |
|------|-----------|
| $\sigma^3_{ij}$ | $\left[(0.809 V_{c,i}^{1/3})(0.809 V_{c,j}^{1/3})\right]^{3/2}$ |
| $(\epsilon/k)_{ij}^0$ | $\sigma^3_{ij} \cdot \sqrt{(T_{c,i}/1.2593)(T_{c,j}/1.2593)}$ |
| $\omega_{ij}^0$ | $\sigma^3_{ij} \cdot (\omega_i + \omega_j)/2$ |
| $M_{ij}^0$ | $(0.809 V_{c,i}^{1/3})(0.809 V_{c,j}^{1/3}) \cdot \sqrt{(T_{c,i}/1.2593)(T_{c,j}/1.2593)} \cdot \sqrt{2M_i M_j/(M_i+M_j)}$ |
| $\mu_{ij}^0$ | $\mu_i^2 \mu_j^2 / [\sigma^3_{ij} \cdot \sqrt{(\epsilon/k)_{ij}}]$ |
| $\kappa_{ij}$ | $\sqrt{\kappa_i \kappa_j}$ |

#### 런타임 혼합

$$\sigma^3_M = \sum_i \sum_j X_i X_j \sigma^3_{ij}$$

$$(\epsilon/k)_M = \frac{\sum_i \sum_j X_i X_j (\epsilon/k)_{ij}^0}{\sigma^3_M}$$

$$\omega_M = \frac{\sum_i \sum_j X_i X_j \omega_{ij}^0}{\sigma^3_M}$$

$$\sigma_M = (\sigma^3_M)^{1/3}, \quad V_{c,M} = (\sigma_M / 0.809)^3, \quad T_{c,M} = 1.2593(\epsilon/k)_M$$

$$M_M = \sqrt{\frac{\sum_i \sum_j X_i X_j M_{ij}^0}{(\epsilon/k)_M \cdot \sigma_M^2}}$$

$$\mu_{i,M} = \left(\sum_i \sum_j X_i X_j \mu_{ij}^0\right)^{1/4}$$

$$\kappa_{i,M} = \left(\sum_i \sum_j X_i X_j \kappa_{ij}\right)^{1/2}$$

### 5.3 Takahashi 이성분 확산 혼합 규칙

#### 사전 계산 행렬

| 행렬 | 결합 규칙 |
|------|-----------|
| $M_{ij}^D$ | $1/M_i + 1/M_j$ (나중에 $\sqrt{2/(1/M_i + 1/M_j)}$로 사용) |
| $\sigma_{ij}^D$ | $(\sigma_{v,i})^{1/3} + (\sigma_{v,j})^{1/3}$ |

#### 런타임 혼합

$$T_{c,ij} = \frac{X_i T_{c,i} + X_j T_{c,j}}{X_i + X_j}, \quad P_{c,ij} = \frac{X_i P_{c,i} + X_j P_{c,j}}{X_i + X_j}$$

이 값들은 Takahashi $\phi(P_r, T_r)$ 보정 함수의 입력으로 사용된다.

In [ ]:
render_mermaid("""
flowchart TD
    subgraph INIT["초기화 (1회) — 사전 계산 행렬"]
        direction TB
        SP["종 리스트\nTc, Pc, Vc, omega,\nmiui, kappai, sigmvi"]
        
        SRK_MAT["<b>SRK 행렬</b>\nCOEF1[i][j] = sqrt(ai*aj)*(1+Si)*(1+Sj)\nCOEF2[i][j], COEF3[i][j]\nBM[i] = 0.08664*R*Tc_i/Pc_i"]
        
        CHUNG_MAT["<b>Chung 행렬</b>\nSIGMA3M[i][j]\nEPSILONKM0[i][j]\nOMEGAM0[i][j]\nMM0[i][j], MIUIM0[i][j]\nKAPPAIM[i][j]"]
        
        TAKA_MAT["<b>Takahashi 행렬</b>\nMMD[i][j] = 1/Mi + 1/Mj\nSIGMD[i][j] = sigmvi^1/3 + sigmvj^1/3"]
        
        SP --> SRK_MAT
        SP --> CHUNG_MAT
        SP --> TAKA_MAT
    end

    subgraph RUNTIME["런타임 (매 셀/면마다)"]
        direction TB
        YI["Yi (질량분율)\n↓ Xi (몰분율) 변환"]
        
        SRK_MIX["<b>SRK 혼합</b>\nbM = sum(Xi*BM[i])\ncoef1M = sum(Xi*Xj*COEF1[i][j])\ncoef2M, coef3M 동일\n→ updateEoS(bM, coef1M, ...)"]
        
        CHUNG_MIX["<b>Chung 혼합</b>\nsigma3M = sum(Xi*Xj*SIGMA3M[i][j])\nepsilonkM = sum(...)/sigma3M\nomegaM, MM, miuiM, kappaiM\n→ updateTRANS(...)"]
        
        TAKA_MIX["<b>Takahashi 혼합</b>\nTcij = (Xi*Tci+Xj*Tcj)/(Xi+Xj)\nPcij 동일\n→ Tcmd, Pcmd 리스트 업데이트"]
        
        YI --> SRK_MIX
        YI --> CHUNG_MIX
        YI --> TAKA_MIX
    end

    subgraph EVAL["물성 평가"]
        RHO["rho(p,T)\nZ → rho = p/(ZRT)"]
        MU["mu(p,T)\nChung 점성"]
        KAPPA["kappa(p,T)\nChung 열전도도"]
        DIMIX["Dimix(i, p, T)\nFuller + Takahashi"]
    end

    SRK_MIX --> RHO
    CHUNG_MIX --> MU
    CHUNG_MIX --> KAPPA
    TAKA_MIX --> DIMIX

    style INIT fill:#E8F5E9,stroke:#2E7D32
    style RUNTIME fill:#E3F2FD,stroke:#1565C0
    style EVAL fill:#FFF3E0,stroke:#E65100
    style SRK_MAT fill:#C8E6C9,stroke:#2E7D32
    style CHUNG_MAT fill:#FFCCBC,stroke:#E64A19
    style TAKA_MAT fill:#D1C4E9,stroke:#7B1FA2
    style SRK_MIX fill:#C8E6C9,stroke:#2E7D32
    style CHUNG_MIX fill:#FFCCBC,stroke:#E64A19
    style TAKA_MIX fill:#D1C4E9,stroke:#7B1FA2
""", height=700)

---
## 6. Phase 4 — 템플릿 인스턴스화 및 솔버 통합

### 6.1 신규 파일/디렉토리 구조 (전체)

```
OpenFOAM-13/
├── src/thermophysicalModels/
│   ├── specie/
│   │   ├── rfSpecie/                          ← [신규] 실유체 종 기본 클래스
│   │   │   ├── rfSpecie.H
│   │   │   ├── rfSpecieI.H
│   │   │   └── rfSpecie.C
│   │   ├── equationOfState/
│   │   │   └── SRKGas/                        ← [신규] SRK 상태방정식
│   │   │       ├── SRKGas.H
│   │   │       ├── SRKGasI.H
│   │   │       └── SRKGas.C
│   │   ├── transport/
│   │   │   └── chungTransport/                ← [신규] Chung+Takahashi 수송물성
│   │   │       ├── chungTransport.H
│   │   │       ├── chungTransportI.H
│   │   │       └── chungTransport.C
│   │   ├── include/
│   │   │   └── forRealFluidGases.H            ← [신규] 템플릿 인스턴스화 매크로
│   │   └── Make/
│   │       └── files                          ← [수정] 신규 소스 추가
│   │
│   └── multicomponentThermo/
│       └── mixtures/
│           └── SRKchungTakaMixture/           ← [신규] 전용 혼합 클래스
│               ├── SRKchungTakaMixture.H
│               ├── SRKchungTakaMixture.C
│               └── SRKchungTakaMixtureI.H
│
└── tutorials/multicomponentFluid/
    └── SandiaD_LTS_supercritical/             ← [신규] 검증 케이스
```

### 6.2 템플릿 인스턴스화 매크로

**`forRealFluidGases.H`:**

```cpp
#include "rfSpecie.H"
#include "SRKGas.H"
#include "janafThermo.H"
#include "sensibleEnthalpy.H"
#include "sensibleInternalEnergy.H"
#include "chungTransport.H"

// 템플릿 체인:
// chungTransport< species::thermo< janafThermo< SRKGas<rfSpecie> >, sensibleEnthalpy > >

#define forRealFluidEqns(Mu, He, Cp, Macro, Args...)              \
    forThermo(Mu, He, Cp, SRKGas, rfSpecie, Macro, Args)

#define forRealFluidEnergiesAndThermos(Mu, Macro, Args...)         \
    forRealFluidEqns(Mu, sensibleEnthalpy, janafThermo, Macro, Args); \
    forRealFluidEqns(Mu, sensibleInternalEnergy, janafThermo, Macro, Args)

#define forRealFluidGases(Macro, Args...)                          \
    forRealFluidEnergiesAndThermos(chungTransport, Macro, Args)
```

### 6.3 `Make/files` 수정

```make
# 기존 파일들 ...

# RGP-13: Real Fluid Models
rfSpecie/rfSpecie.C
equationOfState/SRKGas/SRKGas.C
transport/chungTransport/chungTransport.C
```

### 6.4 multicomponentThermo 인스턴스화

`psiMulticomponentThermos.C` 또는 별도 파일에 추가:

```cpp
#include "forRealFluidGases.H"
#include "SRKchungTakaMixture.H"
#include "makeFluidMulticomponentThermo.H"

namespace Foam
{
    forRealFluidGases
    (
        makeFluidMulticomponentThermo,
        fluidMulticomponentThermo,
        SRKchungTakaMixture
    );
}
```

### 6.5 multicomponentFluid 솔버 연동

기존 `multicomponentFluid` 솔버는 수정 없이 사용 가능 — 
`fluidMulticomponentThermo::New()`가 런타임 선택 테이블에서 자동으로 
`SRKchungTakaMixture` 타입을 찾는다.

**`constant/physicalProperties` 예시:**

```
thermoType
{
    type            heRhoThermo;          // 또는 hePsiThermo
    mixture         SRKchungTakaMixture;
    transport       chung;
    thermo          janaf;
    equationOfState SRKGas;
    specie          rfSpecie;
    energy          sensibleEnthalpy;
}
```

---
## 7. Phase 5 — 검증 계획

### 7.1 단일 종 검증 (NIST 비교)

| 종 | 조건 | 검증 물성 | NIST 데이터 |
|----|------|-----------|------------|
| N₂ | T=100~500K, P=1~100atm | ρ, Cp, μ, κ | NIST WebBook |
| O₂ | T=100~500K, P=1~100atm | ρ, Cp, μ, κ | NIST WebBook |
| CH₄ | T=100~500K, P=1~100atm | ρ, Cp, μ, κ | NIST WebBook |
| H₂ | T=30~500K, P=1~100atm | ρ, Cp, μ, κ | NIST WebBook |
| H₂O | T=300~1000K, P=1~300atm | ρ, Cp, μ, κ | NIST WebBook |

**허용 오차:**
- ρ: < 5% (임계점 근방 제외)
- Cp: < 10% (임계점 근방 피크 제외)
- μ: < 10%
- κ: < 15%

### 7.2 다성분 혼합물 검증

- **O₂/N₂ 혼합** (공기) — 실험 데이터 풍부
- **CH₄/O₂ 혼합** — 연소 관련
- **H₂/O₂ 혼합** — Oefelein 벤치마크 대상

### 7.3 Oefelein LOx/GH₂ 벤치마크

| 항목 | 값 |
|------|-----|
| 압력 | ~100 atm (초임계) |
| 산화제 | LOx (액체 산소, T < Tc) |
| 연료 | GH₂ (기체 수소, T >> Tc) |
| 형상 | 동축 전단 인젝터 |
| 비교 대상 | 밀도장, 온도장, OH 분포 |

### 7.4 핵심 위험 요소 및 대응

| 위험 | 영향 | 대응 |
|------|------|------|
| Z solver 수렴 실패 (임계점 근방) | 발산 | Newton-Raphson 보조 풀이, 근 선택 로직 강화 |
| Cp 발산 (의사-임계점) | 시간스텝 폭감 | LTS 시간스케일에 Cp 제한 추가 |
| 혼합규칙 수치 불안정 | NaN | 0 체크, floor값 (1e-30) 설정 |
| OFv8→OF13 API 불일치 | 컴파일 실패 | PengRobinsonGas 패턴 철저 준수 |
| 템플릿 조합 폭발 | 컴파일 시간 | forRealFluidGases 매크로로 조합 제한 |

---
## 8. 전체 모듈 상호작용 다이어그램

In [ ]:
render_mermaid("""
flowchart LR
    subgraph SOLVER["multicomponentFluid 솔버"]
        SPECIES["<b>종 방정식</b>\n∂(ρYi)/∂t + ..."]
        ENERGY["<b>에너지 방정식</b>\n∂(ρhe)/∂t + ..."]
        MOMENTUM["<b>운동량 방정식</b>\n∂(ρU)/∂t + ..."]
    end

    subgraph MIXTURE["SRKchungTakaMixture"]
        direction TB
        MR_SRK["<b>SRK 혼합</b>\n2차 결합규칙\naα_mix, b_mix"]
        MR_CHUNG["<b>Chung 혼합</b>\nL-J 파라미터\nσ, ε/k, ω, M"]
        MR_TAKA["<b>Takahashi 혼합</b>\nTc_ij, Pc_ij\nσ_ij, M_ij"]
    end

    subgraph EOS_MOD["SRKGas EOS"]
        SRK_Z["Z³ - Z² + ...\nCardano solver"]
        SRK_RHO["ρ = p/(ZRT)"]
        SRK_H["h_dep, Cp_dep\ne_dep, Cv_dep"]
    end

    subgraph TRANS_MOD["chungTransport"]
        CH_MU["μ(p,T)\nChung 1988\nη_k + η_p"]
        CH_KAP["κ(p,T)\nChung 1988\nλ_k + λ_p"]
        CH_DIM["D_mix(p,T)\nFuller + Takahashi\nφ(Pr,Tr) 보정"]
    end

    JANAF["<b>janafThermo</b>\nCp_ideal(T)\nJANAF 다항식"]

    RFSP["<b>rfSpecie</b>\nTc, Pc, Vc, ω\nμi, κi, σvi"]

    %% 데이터 흐름
    SPECIES -->|"Yi"| MIXTURE
    MIXTURE -->|"a,b 혼합"| EOS_MOD
    MIXTURE -->|"σ,ε/k 혼합"| TRANS_MOD
    
    EOS_MOD -->|"ρ, h_dep"| ENERGY
    EOS_MOD -->|"ρ, psi"| MOMENTUM
    
    TRANS_MOD -->|"μ → τ"| MOMENTUM
    TRANS_MOD -->|"κ → q"| ENERGY
    TRANS_MOD -->|"D_mix → ji"| SPECIES

    JANAF -->|"Cp_ideal"| SRK_H
    JANAF -->|"Cv_ideal"| CH_KAP
    RFSP -->|"임계점 물성"| MIXTURE

    style SOLVER fill:#E3F2FD,stroke:#1565C0
    style MIXTURE fill:#FFF3E0,stroke:#E65100
    style EOS_MOD fill:#E8F5E9,stroke:#2E7D32
    style TRANS_MOD fill:#FCE4EC,stroke:#C2185B
    style JANAF fill:#F5F5F5,stroke:#616161
    style RFSP fill:#BBDEFB,stroke:#1565C0
""", height=500)

---
## 9. 구현 우선순위 체크리스트

### Phase 1: 기반 (필수)
- [ ] `rfSpecie` 클래스 생성 (specie 확장, Tc/Pc/Vc/omega/miui/kappai/sigmvi)
- [ ] `rfSpecie` operator+=, operator*= 구현
- [ ] `rfSpecie` 딕셔너리 읽기/쓰기
- [ ] `SRKGas` 클래스 생성 (PengRobinsonGas 패턴)
- [ ] SRK Z 풀이 (Cardano, 3근 선택 로직)
- [ ] SRK rho, h, Cp, psi, Z, CpMCv 구현 (OFv8 참조)
- [ ] SRK e, Cv, alphav 완전 구현 (OFv8 미구현분 보완)
- [ ] SRK updateEoS() 메서드 (혼합 파라미터 외부 주입)
- [ ] **단일종 NIST 검증** (N₂, O₂, CH₄)

### Phase 2: 수송물성 (필수)
- [ ] `chungTransport` 클래스 생성
- [ ] calculate() — T*, mur, eta0, Y, G1 중간변수
- [ ] mu(p,T) — Chung 점성 (A계수 테이블 포함)
- [ ] kappa(p,T) — Chung 열전도도 (B계수 테이블 포함)
- [ ] phi(Pr, Tr) — Takahashi 보정계수 (보간 테이블)
- [ ] Dimix(speciei, p, T) — 혼합-평균 확산계수
- [ ] updateTRANS() 메서드 (혼합 파라미터 외부 주입)
- [ ] 확산계수용 리스트 멤버 (Ymd_, Xmd_, Tcmd_, Pcmd_, Mmd_, sigmd_)
- [ ] **단일종 수송물성 NIST 검증**

### Phase 3: 혼합규칙 (핵심)
- [ ] `SRKchungTakaMixture` 클래스 생성
- [ ] 사전 계산 행렬 초기화 (COEF1~3, SIGMA3M, EPSILONKM0, ...)
- [ ] 런타임 SRK 파라미터 혼합 (quadratic mixing → updateEoS)
- [ ] 런타임 Chung 파라미터 혼합 (L-J mixing → updateTRANS)
- [ ] 런타임 Takahashi 이성분 혼합 (Tcij, Pcij → Dimix)
- [ ] thermoMixture / transportMixture 인터페이스 구현
- [ ] **다성분 혼합물 검증**

### Phase 4: 통합 (필수)
- [ ] `forRealFluidGases.H` 매크로 작성
- [ ] `Make/files` 수정
- [ ] 템플릿 인스턴스화 + 컴파일 확인
- [ ] `constant/physicalProperties` 딕셔너리 형식 확정
- [ ] multicomponentFluid 솔버와 연동 테스트

### Phase 5: 검증 (필수)
- [ ] NIST 다성분 물성 비교 (O₂/N₂, CH₄/O₂, H₂/O₂)
- [ ] Oefelein LOx/GH₂ 벤치마크 케이스 구성
- [ ] 결과 비교 및 문서화

---

## 10. 핵심 참조 파일 경로 요약

| 참조 대상 | 경로 |
|-----------|------|
| **OF-13 PengRobinsonGas** | `OpenFOAM-13/src/thermophysicalModels/specie/equationOfState/PengRobinsonGas/` |
| **OF-13 sutherlandTransport** | `OpenFOAM-13/src/thermophysicalModels/specie/transport/sutherland/` |
| **OF-13 forGases.H** | `OpenFOAM-13/src/thermophysicalModels/specie/include/forGases.H` |
| **OF-13 multicomponentMixture** | `OpenFOAM-13/src/thermophysicalModels/multicomponentThermo/mixtures/multicomponentMixture/` |
| **OFv8 SRK** | `references/realFluidFoam-8/src/thermophysicalModels/specie/equationOfState/soaveRedlichKwong/` |
| **OFv8 Chung+Takahashi** | `references/realFluidFoam-8/src/thermophysicalModels/specie/transport/chungTaka/` |
| **OFv8 SRKchungTakaMixture** | `references/realFluidFoam-8/src/thermophysicalModels/reactionThermo/mixtures/SRKchungTakaMixture/` |
| **OFv8 rfSpecie** | `references/realFluidFoam-8/src/thermophysicalModels/specie/rfSpecie/` |
| **OFv8 매크로** | `references/realFluidFoam-8/src/thermophysicalModels/specie/include/forSRKchungTakaGases.H` |